# Groceries - Data Preparation and Transformation with Restrictions

This file is used to prepare and transform the groceries data. The following restrictions apply to this file:

## Restrictions:

1. Baskets with less than 30 items
2. Customers with less than 5 baskets

## Main Process and Steps:

### 1. Parameter Setup and Data Overview:
Set up the packages, path, and dataset name. It also includes an overview of the data, such as the date period of the data.

### 2. Indexing of Items:
Create indices to represent the items and a mapping table for reference.

### 3. Indexing of Customers:
Create indices to represent the customer numbers and a mapping table for reference.

### 4. Baskets for Each Customer and Items in Each Basket:
By utilizing timestamps, it is possible to determine which items were purchased together, group them into baskets, and identify the number of baskets each customer has.

### 5. Apply Restrictions:
Apply the specified restrictions to the data.

### 6. Separate Train and Test Datasets:
Split the dataset so that the number of baskets in the training set is equal to the number of baskets in the testing set. If a customer has an odd number of baskets, delete the latest date (last row).

### 7. Generate 3 Files as Input for the Model:
Generate the following files: `train_u2b.txt`, `train_b2i.txt`, and `test_b2i.txt`.

## Coding

### 1. Parameter Setup and Data Overview:

In [57]:
# Import packages
import pandas as pd
import numpy as np
import random

In [58]:
# Set path
# Set path
path_name = '../../DataPreparationandTransformation/multimodal_data\data\multimodalwithres'

In [59]:
# Set and read dataset
df = pd.read_csv(path_name + '/Multimodal.csv')
df.head(5)

,LOC_ID,CUSTOMER_ID,TX_ID,TX_DATE,TX_TME,ITEM_ID,SUBGROUP_ID,NET_SALES_UNITS,NET_SALES_AMT
0,8,39260,748538,3/3/19,153300,6100,459,0.949816,3.854712
1,8,30743,876237,3/3/19,201000,1746,974,1.180286,1.834784
2,8,30743,876237,3/3/19,201000,1746,974,1.092798,1.699623
3,8,28346,746752,3/3/19,161400,1315,795,1.039463,2.955283
4,8,77899,925250,3/3/19,154100,8854,699,0.862407,3.591753


In [60]:
# The date period of the data
# Convert 'TX_DATE' column to datetime format
def parse_date(date_str):
    for fmt in ('%m/%d/%Y', '%m/%d/%y'):  # List formats to try
        try:
            return pd.to_datetime(date_str, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['TX_DATE'] = df['TX_DATE'].apply(parse_date)

# Find first and last dates of data
first_date = df['TX_DATE'].min()
last_date = df['TX_DATE'].max()

print(f"First date in the dataset: {first_date}")
print(f"Last date in the dataset: {last_date}")

First date in the dataset: 2019-03-03 00:00:00
Last date in the dataset: 2020-02-21 00:00:00


### 2. Indexing of Items:

In [61]:
# Factorize the ITEM_ID column
df['item_no'], item_labels = pd.factorize(df['ITEM_ID'])

# Create a mapping table
item_mapping = pd.DataFrame({
    'ITEM_ID': item_labels,
    'item_no': range(len(item_labels))
})

# Drop the original ITEM_ID column
df = df.drop(columns=['LOC_ID', 'TX_ID', 'TX_TME', 'ITEM_ID', 'SUBGROUP_ID', 'NET_SALES_UNITS', 'NET_SALES_AMT'])
df

,CUSTOMER_ID,TX_DATE,item_no
0,39260,2019-03-03,0
1,30743,2019-03-03,1
2,30743,2019-03-03,1
3,28346,2019-03-03,2
4,77899,2019-03-03,3
...,...,...,...
790422,88580,2020-02-20,807
790423,20238,2020-02-20,113
790424,20238,2020-02-20,113
790425,20238,2020-02-20,916


### 3. Indexing of Customers:

In [62]:
# Factorize the CUSTOMER_ID column
df['uid'], item_labels = pd.factorize(df['CUSTOMER_ID'])

# Create a mapping table
item_mapping = pd.DataFrame({
    'CUSTOMER_ID': item_labels,
    'uid': range(len(item_labels))
})

# Drop the original CUSTOMER_ID column
df = df.drop(columns=['CUSTOMER_ID'])
df

,TX_DATE,item_no,uid
0,2019-03-03,0,0
1,2019-03-03,1,1
2,2019-03-03,1,1
3,2019-03-03,2,2
4,2019-03-03,3,3
...,...,...,...
790422,2020-02-20,807,10953
790423,2020-02-20,113,6477
790424,2020-02-20,113,6477
790425,2020-02-20,916,6477


### 4. Baskets for Each Customer and Items in Each Basket:

In [63]:
# Convert the 'TX_DATE' column to datetime format
df['TX_DATE'] = pd.to_datetime(df['TX_DATE'], format='%d-%m-%Y')

# Group data by 'uid' and 'Date' to create baskets for each customer and items in each basket
grouped_df = df.groupby(['uid', 'TX_DATE'])['item_no'].apply(list).reset_index()
expanded_df = grouped_df['item_no'].apply(pd.Series).rename(columns=lambda x: str(x+1))
expanded_df = expanded_df.fillna('')
for col in expanded_df.columns[:]:
    expanded_df[col] = expanded_df[col].apply(lambda x: str(int(x)) if x != '' else x)

final_df = pd.concat([grouped_df[['uid', 'TX_DATE']], expanded_df], axis=1)
final_df.rename(columns={'TX_DATE': 'Date'}, inplace=True)

final_df

,uid,Date,1,2,3,4,5,6,7,8,...,516,517,518,519,520,521,522,523,524,525
0,0,2019-03-03,0,24,24,26,27,28,46,47,...,,,,,,,,,,
1,0,2019-03-24,28,28,583,583,99,99,24,24,...,,,,,,,,,,
2,0,2019-04-07,28,24,587,2420,,,,,...,,,,,,,,,,
3,0,2019-04-14,152,1163,1163,1163,397,397,152,152,...,,,,,,,,,,
4,0,2019-04-24,35,35,35,328,955,955,955,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239255,37346,2020-02-20,693,,,,,,,,...,,,,,,,,,,
239256,37347,2020-02-20,410,410,410,410,,,,,...,,,,,,,,,,
239257,37348,2020-02-21,3888,,,,,,,,...,,,,,,,,,,
239258,37349,2020-02-21,433,731,731,731,151,1325,1325,1325,...,,,,,,,,,,


### 5. Apply Restrictions:

In [64]:
# Baskets with less than 30 items
# Define a function to count items
def count_valid_values(row):
    return row.replace('', pd.NA).dropna().astype(bool).sum()

final_df = final_df.assign(valid_count=final_df.apply(count_valid_values, axis=1))\
                    .query('valid_count <= 31')

# Trim columns to the maximum number of valid values
max_valid_count = final_df['valid_count'].max()
final_df = final_df.drop(columns=['valid_count'])

def trim_columns(row, max_count):
    valid_values = row[row != '']
    if len(valid_values) > max_count:
        trimmed = valid_values.iloc[:max_count].tolist()
    else:
        trimmed = valid_values.tolist()
    return trimmed + [''] * (max_count - len(trimmed))

trimmed_data = final_df.apply(lambda row: trim_columns(row, max_valid_count), axis=1)
df_trimmed = pd.DataFrame(trimmed_data.tolist(), index=final_df.index)
# First rename the first two columns as you've already done
final_df = df_trimmed.rename(columns={df_trimmed.columns[0]: 'uid', df_trimmed.columns[1]: 'Date'})
# Now rename the remaining columns as sequential numbers
remaining_cols = final_df.columns[2:]  # Get all columns after the first two
new_names = {col: str(i+1) for i, col in enumerate(remaining_cols)}  # Create mapping of old->new names
final_df = final_df.rename(columns=new_names)  # Apply the renaming
# Display the result
final_df

,uid,Date,1,2,3,4,5,6,7,8,...,20,21,22,23,24,25,26,27,28,29
0,0,2019-03-03,0,24,24,26,27,28,46,47,...,,,,,,,,,,
1,0,2019-03-24,28,28,583,583,99,99,24,24,...,,,,,,,,,,
2,0,2019-04-07,28,24,587,2420,,,,,...,,,,,,,,,,
3,0,2019-04-14,152,1163,1163,1163,397,397,152,152,...,,,,,,,,,,
4,0,2019-04-24,35,35,35,328,955,955,955,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239255,37346,2020-02-20,693,,,,,,,,...,,,,,,,,,,
239256,37347,2020-02-20,410,410,410,410,,,,,...,,,,,,,,,,
239257,37348,2020-02-21,3888,,,,,,,,...,,,,,,,,,,
239258,37349,2020-02-21,433,731,731,731,151,1325,1325,1325,...,,,,,,,,,,


In [65]:
# Customers with less than 5 baskets
# Keep the first 10 rows
final_df = final_df.groupby('uid').apply(lambda x: x.head(10)).reset_index(drop=True)
final_df

C:\Users\admin\AppData\Local\Temp\ipykernel_12236\3845527340.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = final_df.groupby('uid').apply(lambda x: x.head(10)).reset_index(drop=True)


,uid,Date,1,2,3,4,5,6,7,8,...,20,21,22,23,24,25,26,27,28,29
0,0,2019-03-03,0,24,24,26,27,28,46,47,...,,,,,,,,,,
1,0,2019-03-24,28,28,583,583,99,99,24,24,...,,,,,,,,,,
2,0,2019-04-07,28,24,587,2420,,,,,...,,,,,,,,,,
3,0,2019-04-14,152,1163,1163,1163,397,397,152,152,...,,,,,,,,,,
4,0,2019-04-24,35,35,35,328,955,955,955,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145462,37346,2020-02-20,693,,,,,,,,...,,,,,,,,,,
145463,37347,2020-02-20,410,410,410,410,,,,,...,,,,,,,,,,
145464,37348,2020-02-21,3888,,,,,,,,...,,,,,,,,,,
145465,37349,2020-02-21,433,731,731,731,151,1325,1325,1325,...,,,,,,,,,,


### 6. Separate Train and Test Datasets:

In [66]:
# ===== Step 1: Global Train/Test Split =====
train_rows = []
test_rows = []
global_basket_number = 0

# Go through each user
for uid, user_data in final_df.groupby('uid'):
    user_data_sorted = user_data  # assume already sorted by date

    if len(user_data_sorted) < 2:
        continue  # Skip users with less than 2 baskets

    # Split
    train_data = user_data_sorted.iloc[:-1].copy()
    test_data = user_data_sorted.iloc[-1:].copy()

    # Assign basket_number (global continuous index)
    train_data['basket_number'] = range(global_basket_number, global_basket_number + len(train_data))
    global_basket_number += len(train_data)

    test_data['basket_number'] = [global_basket_number]
    global_basket_number += 1

    train_rows.append(train_data)
    test_rows.append(test_data)

# Merge all users' train and test data
train_df = pd.concat(train_rows).reset_index(drop=True)
test_df = pd.concat(test_rows).reset_index(drop=True)


In [67]:
print("Training Data:")
print(train_df.head())

Training Data:
   uid       Date    1     2     3     4    5    6    7    8  ... 21 22 23 24  \
0    0 2019-03-03    0    24    24    26   27   28   46   47  ...               
1    0 2019-03-24   28    28   583   583   99   99   24   24  ...               
2    0 2019-04-07   28    24   587  2420                      ...               
3    0 2019-04-14  152  1163  1163  1163  397  397  152  152  ...               
4    0 2019-04-24   35    35    35   328  955  955  955       ...               

  25 26 27 28 29 basket_number  
0                            0  
1                            1  
2                            2  
3                            3  
4                            4  

[5 rows x 32 columns]


In [68]:
print("\nTesting Data:")
print(test_df.head())



Testing Data:
   uid       Date     1    2    3    4    5     6    7    8  ... 21 22 23 24  \
0    0 2019-09-15   452  583  583  452  452  4429  452  452  ...               
1    1 2019-05-15    91    1    1                            ...               
2    2 2019-12-13   854  715                                 ...               
3    3 2019-07-12   819                                      ...               
4    4 2019-11-19  4579                                      ...               

  25 26 27 28 29 basket_number  
0                            9  
1                           19  
2                           29  
3                           39  
4                           49  

[5 rows x 32 columns]


### 7. Generate 3 Files as Input for the Model:

In [69]:
# train_b2i
train_b2i = train_df.drop(columns=['uid'])

# Assign 'basket_number' as the first column
columns = ['basket_number'] + [col for col in train_b2i.columns if col != 'basket_number']
train_b2i = train_b2i[columns]

# Replace NaN with empty strings
train_b2i = train_b2i.fillna('')

# Convert all values to strings, handling empty values and non-string types
def clean_and_convert(value):
    if pd.isna(value) or value == '':
        return ''
    try:
        if isinstance(value, (str, int)):
            return str(int(value))
        elif isinstance(value, float) and not pd.isna(value):
            return str(int(value))
        else:
            return ''
    except ValueError:
        return ''

for col in train_b2i.columns[1:]:
    train_b2i[col] = train_b2i[col].apply(clean_and_convert)

# Formatting data
def format_row(row):
    return ' '.join(str(x) for x in row if x != '')

# Add timestamps
train_b2i['timestamp'] = train_df['Date'].dt.strftime('%Y%m%d')

formatted_rows = train_b2i.apply(lambda row: ' '.join(str(x).strip() for x in row if x != ''), axis=1)

# formatted_rows = train_b2i.apply(format_row, axis=1)
def validate_timestamp(row):
    try:

        return row
    except ValueError:
        return None

formatted_rows = formatted_rows.apply(validate_timestamp).dropna()  


# Export the data as a text file without column names
with open(f'{path_name}/train_b2i.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}\\train_b2i.txt')

Data exported to ../../DataPreparationandTransformation/multimodal_data\data\multimodalwithres\train_b2i.txt


In [70]:
# Test set - test_b2i
# Drop the original 'uid' column 
test_b2i = test_df.drop(columns=['uid'])

# Assign 'basket_number' as the first column
columns = ['basket_number'] + [col for col in test_b2i.columns if col != 'basket_number']
test_b2i = test_b2i[columns]

# Replace NaN with empty strings
test_b2i = test_b2i.fillna('')

# Convert all values to strings, handling empty values and non-string types
for col in test_b2i.columns[1:]:  
    test_b2i[col] = test_b2i[col].apply(clean_and_convert)

# Formatting function to join values with space, ignoring empty strings
def format_row(row):
    return ' '.join(str(x) for x in row if x != '')

# Add timestamps
test_b2i['timestamp'] = test_df['Date'].dt.strftime('%Y%m%d')

formatted_rows = test_b2i.apply(lambda row: ' '.join(str(x).strip() for x in row if x != ''), axis=1)
formatted_rows = formatted_rows.apply(validate_timestamp).dropna()  

# Formatting data
# formatted_rows = test_b2i.apply(format_row, axis=1)

# Export the data as a text file without column names
with open(f'{path_name}/test_b2i.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}\\test_b2i.txt')

Data exported to ../../DataPreparationandTransformation/multimodal_data\data\multimodalwithres\test_b2i.txt


In [71]:
# ===== Step 4: train_u2b =====

# Group and expand: uid -> basket list
grouped_df = train_df.groupby('uid')['basket_number'].apply(list).reset_index()

# Expand baskets
expanded_df = grouped_df['basket_number'].apply(pd.Series)
expanded_df.columns = [f'basket_{i+1}' for i in expanded_df.columns]

# Concatenate uid + baskets
train_u2b = pd.concat([grouped_df['uid'], expanded_df], axis=1)

# Replace NaN
train_u2b = train_u2b.fillna('')

# Clean and convert
for col in train_u2b.columns[1:]:  # Skip uid
    train_u2b[col] = train_u2b[col].apply(clean_and_convert)

# Formatting
formatted_rows = train_u2b.apply(format_row, axis=1)

# Export to txt
with open(f'{path_name}/train_u2b.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}/train_u2b.txt')


Data exported to ../../DataPreparationandTransformation/multimodal_data\data\multimodalwithres/train_u2b.txt


In [72]:
# ===== Step 5: test_u2b =====

# Only take ['uid', 'basket_number']
test_u2b = test_df[['uid', 'basket_number']].copy()

# Replace NaN
test_u2b = test_u2b.fillna('')

# Clean and convert
for col in test_u2b.columns[1:]:  # Skip uid
    test_u2b[col] = test_u2b[col].apply(clean_and_convert)

# Formatting
formatted_rows = test_u2b.apply(format_row, axis=1)

# Export to txt
with open(f'{path_name}/test_u2b.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}/test_u2b.txt')



Data exported to ../../DataPreparationandTransformation/multimodal_data\data\multimodalwithres/test_u2b.txt
